# Iterative code refinement: an LLM feedback loop

Companion notebook for **lesson 5.1**. The LLM writes a function, an external tool runs it against tests, and the test results go back into the next prompt, until every test passes or the iteration limit is reached.

**Run it in Colab, setup cell first.** It uses Colab's built-in models (no API key); uncomment a `MODEL = ...` line there to choose one.

In [ ]:
# Setup: Colab's built-in models, no API key needed. Run this cell first.
from google.colab import ai
print("Available models:", ", ".join(ai.list_models()))

MODEL = None                              # Colab's default; uncomment a line below to choose
# MODEL = "google/gemini-2.5-flash"
# MODEL = "google/gemini-2.5-pro"
# MODEL = "google/gemini-3.8-flash"

def get_completion(messages):
    """Send chat messages to the model. Colab's generate_text takes one prompt, so roles are inlined."""
    prompt = "\n\n".join(f"{m['role'].upper()}:\n{m['content']}" for m in messages) + "\n\nASSISTANT:\n"
    return ai.generate_text(prompt, model_name=MODEL) if MODEL else ai.generate_text(prompt)

## 1 · The task, the tests, and the test runner

The test runner is the **external tool**: it runs the generated code in a separate process with a timeout, and reports input, expected and actual output for every failure. Detailed feedback is what lets the model fix its own code.

In [1]:
import json, re, subprocess, sys, tempfile, textwrap

TASK = ("Write a Python function called sort_numbers that takes a list of integers and "
        "returns a new list with the numbers sorted in ascending order.")

TEST_CASES = [
    {"input": [3, 1, 2],      "expected": [1, 2, 3]},
    {"input": [],             "expected": []},
    {"input": [5, -1, 5, 0],  "expected": [-1, 0, 5, 5]},
]

HARNESS = textwrap.dedent("""
    import json, sys
    ns = {}
    exec(open(sys.argv[1]).read(), ns)
    results = []
    for case in json.loads(sys.argv[2]):
        try:
            actual = ns["sort_numbers"](list(case["input"]))
            results.append({**case, "actual": actual, "passed": actual == case["expected"]})
        except Exception as e:
            results.append({**case, "actual": f"{type(e).__name__}: {e}", "passed": False})
    print(json.dumps(results))
""")

def strip_fences(text: str) -> str:
    """Models usually wrap code in a Markdown fence; remove it before running."""
    return re.sub(r"^\s*`{3}(?:python)?\s*\n|\n?`{3}\s*$", "", text.strip())

def run_tests(code: str) -> list[dict]:
    """The external tool: run the code in a separate process, with a timeout."""
    with tempfile.NamedTemporaryFile("w", suffix=".py", delete=False) as f:
        f.write(code)
    try:
        out = subprocess.run([sys.executable, "-c", HARNESS, f.name, json.dumps(TEST_CASES)],
                             capture_output=True, text=True, timeout=5)
    except subprocess.TimeoutExpired:
        return [{**c, "actual": "timed out", "passed": False} for c in TEST_CASES]
    if out.returncode != 0:                       # didn't even load: syntax error, crash
        error = out.stderr.strip().splitlines()[-1] if out.stderr.strip() else "unknown error"
        return [{**c, "actual": error, "passed": False} for c in TEST_CASES]
    return json.loads(out.stdout)

def feedback_text(results: list[dict]) -> str:
    """Detailed feedback: input, expected and actual for every failure."""
    lines = [f"{sum(r['passed'] for r in results)}/{len(results)} tests passed."]
    for r in results:
        if not r["passed"]:
            lines.append(f"FAIL input={r['input']} expected={r['expected']} actual={r['actual']}")
    return "\n".join(lines)

## 2 · The feedback, on code with typical bugs

Before involving a model, run the test runner on four versions of the function, each with a bug models really make. This is exactly the feedback the model receives in the loop below.

In [2]:
attempts = {
    "returns nums.sort()":       "def sort_numbers(nums):\n    return nums.sort()",
    "sorts in reverse":          "def sort_numbers(nums):\n    return sorted(nums, reverse=True) if nums else []",
    "set() drops duplicates":    "def sort_numbers(nums):\n    return sorted(set(nums))",
    "correct":                   "def sort_numbers(nums):\n    return sorted(nums)",
    "syntax error":              "def sort_numbers(nums:\n    return sorted(nums)",
    "infinite loop":             "def sort_numbers(nums):\n    while True:\n        pass",
}
for label, attempt in attempts.items():
    print(f"── {label}\n{feedback_text(run_tests(attempt))}\n")

── returns nums.sort()
0/3 tests passed.
FAIL input=[3, 1, 2] expected=[1, 2, 3] actual=None
FAIL input=[] expected=[] actual=None
FAIL input=[5, -1, 5, 0] expected=[-1, 0, 5, 5] actual=None

── sorts in reverse
1/3 tests passed.
FAIL input=[3, 1, 2] expected=[1, 2, 3] actual=[3, 2, 1]
FAIL input=[5, -1, 5, 0] expected=[-1, 0, 5, 5] actual=[5, 5, 0, -1]

── set() drops duplicates
2/3 tests passed.
FAIL input=[5, -1, 5, 0] expected=[-1, 0, 5, 5] actual=[-1, 0, 5]

── correct
3/3 tests passed.

── syntax error
0/3 tests passed.
FAIL input=[3, 1, 2] expected=[1, 2, 3] actual=SyntaxError: '(' was never closed
FAIL input=[] expected=[] actual=SyntaxError: '(' was never closed
FAIL input=[5, -1, 5, 0] expected=[-1, 0, 5, 5] actual=SyntaxError: '(' was never closed



── infinite loop
0/3 tests passed.
FAIL input=[3, 1, 2] expected=[1, 2, 3] actual=timed out
FAIL input=[] expected=[] actual=timed out
FAIL input=[5, -1, 5, 0] expected=[-1, 0, 5, 5] actual=timed out



## 3 · The loop

Generate → run the tests → if anything fails, send the previous code **and** the feedback back → repeat, until all tests pass or the iteration limit is hit. The history list is the monitoring: one line per iteration.

In [3]:
def refine_until_passing(max_iterations: int = 5):
    code = strip_fences(get_completion([{"role": "user", "content": TASK}]))   # step 1: first attempt
    history = []
    for i in range(1, max_iterations + 1):
        results = run_tests(code)                                              # step 2: run the tests
        passed = sum(r["passed"] for r in results)
        history.append(f"Iteration {i}: {passed}/{len(results)} passed")       # monitoring
        if passed == len(results):
            return code, history                                               # ✅ all tests pass
        code = strip_fences(get_completion([{"role": "user", "content":         # step 3: feed results back
            f"{TASK}\nRe-write the function using the provided feedback.\n\n"
            f"FUNCTION:\n{code}\n\nFEEDBACK:\n{feedback_text(results)}"}]))
    raise RuntimeError("Tests still failing after the iteration limit:\n" + "\n".join(history))

## 4 · Run it with a real model

A strong model often passes this simple task on the first try. To watch the loop work, choose a smaller model in the setup cell, or make the task harder (e.g. "sort by absolute value, ties keep their original order") and add tests for it.

In [ ]:
code, history = refine_until_passing()
print("\n".join(history))
print("\nFinal code:\n" + code)